# 01 Bigram 문자 모델

## 목적
한국어 금융 말뭉치에서 vocabulary, `stoi`, `itos`를 만들고, 현재 문자 하나로 다음 문자를 예측하는 Bigram next-character model을 확인한다.

In [1]:
from pathlib import Path
import sys, torch

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.tokenizer import CharTokenizer, load_text
from src.baselines import BigramNextChar

torch.manual_seed(42)
text = load_text(str(ROOT / 'data/korean_finance_corpus.txt'))
tokenizer = CharTokenizer(text)
print('vocab_size:', tokenizer.vocab_size)
print('stoi sample:', list(tokenizer.stoi.items())[:8])
print('itos sample:', [(i, tokenizer.itos[i]) for i in range(8)])

vocab_size: 370
stoi sample: [('\n', 0), (' ', 1), (',', 2), ('-', 3), ('.', 4), ('G', 5), ('K', 6), ('L', 7)]
itos sample: [(0, '\n'), (1, ' '), (2, ','), (3, '-'), (4, '.'), (5, 'G'), (6, 'K'), (7, 'L')]


## 핵심 코드
`BigramNextChar`는 `(V, V)` embedding table을 사용한다. 입력 문자 id 하나가 다음 문자 전체 어휘에 대한 logits가 된다.

In [2]:
ids = torch.tensor(tokenizer.encode(text[:120]), dtype=torch.long)
x, y = ids[:-1], ids[1:]
model = BigramNextChar(tokenizer.vocab_size)
logits, loss = model(x, y)
print('x shape:', tuple(x.shape))
print('logits shape:', tuple(logits.shape))
print('cross entropy:', round(float(loss), 4))

x shape: (119,)
logits shape: (119, 370)
cross entropy: 6.2522


/tmp/ipykernel_44697/3665112907.py:7: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:838.)
  print('cross entropy:', round(float(loss), 4))


In [3]:
opt = torch.optim.AdamW(model.parameters(), lr=0.05)
for _ in range(8):
    _, loss = model(x, y)
    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()

start_id = tokenizer.encode('인')[0]
sample_ids = model.sample(start_id, steps=40)
print(tokenizer.decode(sample_ids))

인객계익익케윤상택피따크르금능문.S목술건K을의시석N운개므티그빅앞처것애긴씩람특


## 이 단계의 한계
Bigram은 직전 문자 하나만 본다. 긴 단어, 문장 구조, 주제 흐름을 거의 기억하지 못한다.

## 다음 단계가 해결하는 점
다음 노트북은 `block_size=3` 문맥을 embedding하고 flatten한 뒤 MLP로 여러 글자 문맥을 사용한다.